<a href="https://colab.research.google.com/github/Sathvikabanavath/GenAI-Tasks/blob/main/Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai faiss-cpu tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.1 MB/s eta 0:00:00


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"

In [ ]:
documents = [
    "RAG stands for Retrieval Augmented Generation.",
    "In RAG, relevant documents are retrieved from a database before generating an answer.",
    "Vector databases store embeddings for fast similarity search.",
    "OpenAI models can generate answers using retrieved context."
]

In [ ]:
import faiss
import numpy as np
from openai import OpenAI

client = OpenAI()

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(response.data[0].embedding, dtype="float32")

# Create embeddings
embeddings = np.array([get_embedding(doc) for doc in documents])

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [ ]:
def retrieve_context(query, k=2):
    query_embedding = get_embedding(query).reshape(1, -1)
    distances, indices = index.search(query_embedding, k)
    return [documents[i] for i in indices[0]]

In [ ]:
def ask_rag(question):
    context_chunks = retrieve_context(question)

    context = "\n".join(context_chunks)

    prompt = f"""
You are a helpful assistant.
Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [ ]:
question = "What is Retrieval Augmented Generation?"
answer = ask_rag(question)

print("Answer:")
print(answer)

Answer:
Retrieval Augmented Generation is a method where models generate answers using retrieved context.
